## **Exercícios resolvidos das actividades**
* **DISCENTES**
* Adilma Domiana Castro
* Manuel Francisco Domingos César
* Nazaré Inácia Alfredo
* Paulina Caminha Afonso

# Aplicação Estatística a problemas reais da Biologia

Nesta aula vamos analisar o crescimento de **6 microrganismos** em cultura pura, extrair parâmetros biológicos das curvas de crescimento, e aplicar testes estatísticos para comparar os organismos entre si.

**Organismos em estudo:**

| Organismo | Tipo | Patogenicidade | Ambiente natural |
|---|---|---|---|
| *E. coli* K12 | Bactéria Gram− | Comensal | Intestino humano |
| *Salmonella enterica* | Bactéria Gram− | Patogénico | Intestino, alimentos |
| *L. acidophilus* | Bactéria Gram+ | Comensal/probiótico | Intestino, vagina |
| *C. difficile* | Bactéria Gram+ | Patogénico | Intestino (oportunista) |
| *S. cerevisiae* | Levedura | Comensal/probiótico | Intestino humano, alimentos, solo |
| *C. albicans* | Levedura | Patogénico (oportunista) | Intestino, vagina |

## Curvas de Crescimento

O crescimento microbiano segue um padrão **sigmoidal** bem caracterizado, com fases distintas: **lag**, **exponencial**, **estacionária** e **declínio**. A análise quantitativa destas curvas permite extrair parâmetros biológicos que descrevem o comportamento de cada cultura.

---

### Parâmetros-chave

| Parâmetro | Símbolo | Significado biológico | Unidade |
|---|---|---|---|
| Plateau | **A** | Densidade ótica máxima atingida (capacidade de carga) | OD600 |
| Taxa de crescimento máxima | **μ_max** | Inclinação máxima da curva — quão rápido as células se dividem | OD/h |
| Fase lag | **λ** | Tempo que a cultura demora a iniciar o crescimento exponencial | horas |

---

### Porquê estes parâmetros?

Estes três parâmetros permitem responder a questões práticas importantes, como:
- **Segurança alimentar:** Quanto tempo demora um patogénico a começar a multiplicar-se num alimento? (λ)
- **Produção industrial:** Qual o organismo que atinge maior biomassa? (A)
- **Competição entre espécies:** Quem cresce mais rápido e pode dominar uma co-cultura? (μ_max)


In [ ]:
# instalar Pacotes se necessário
#!pip install numpy pandas matplotlib scipy statsmodels

In [ ]:
# Importar Pacotes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# FASE 1 — Carregar e visualizar os dados

Antes de qualquer cálculo, é fundamental **olhar para os dados**. Vamos carregar o ficheiro CSV com as leituras de OD600 ao longo do tempo para as 6 estirpes (3 réplicas cada).

### 1.1 Abrir as medições de crescimento microbiano

O ficheiro `crescimento_microbiano.csv` contém leituras de **densidade ótica a 600 nm (OD600)** ao longo de 24 horas para as 6 estirpes, com 3 réplicas biológicas cada.

A OD600 é uma medida indireta da **concentração de células** em suspensão — quanto mais células, mais luz é dispersa e maior o valor de OD.

**Estrutura dos dados:**
- Cada **linha** = uma réplica de uma estirpe
- Cada **coluna** numérica = um ponto temporal (em horas)
- Colunas `Strain` e `Replicate` = identificação da amostra


In [ ]:
# Carregar o CSV numa dataframe pandas
df = pd.read_csv("crescimento_microbiano.csv")
df.head()

In [ ]:
df["0"].describe()

In [ ]:

df["Strain"].value_counts()

In [ ]:
print("Boas práticas: ")

print("Verifica as primeiras linhas do DataFrame:")
print()
df.shape


#### **Questão**

- Quantas medições temos no total? (linhas × colunas de tempo)
- Os valores de OD no tempo 0 são todos iguais? O que poderia explicar pequenas diferenças?

#### **Respostas**
* Ao todo temos 18 linhas e 22 colunas (linhas × colunas de tempo)
* Não! Isto acontece porque as medições podem ser diversificads pela capacidade dos microrganismos se adaptarem ao meio e assim acabam causando ruidos de leitura?

In [ ]:
# Numero de medições
df.shape

In [ ]:
print("Verificar quais as condições a analisar")
print("Estirpe:")
print(df["Strain"].unique())

#print("Réplicas por estirpe:")
#print(XXX)


In [ ]:
# Todal de estirpes por analisar
estirpes = df["Strain"].unique()
print(f"Total de estirpes: {len(estirpes)}")

replicas_por_estirpe = df["Replicate"].value_counts()
print(f"\nNúmero de réplicas por estirpe: {replicas_por_estirpe.shape[0]}")

#### **Questão**
- No total quantas estirpes vamos analisar?
- E quantas réplicas temos por análise?

#### **Respostas**
- No total vamos analisar 6 estirpes
- Por cada análise temos 3 replicas por analisar

In [ ]:
print("\nEstatísticas descritivas:")
print(df.describe()) #describe

#### **Questão**
* Como varia o desvio padrão com o tempo?

#### **Resposta**
* Ao longo do tempo, o desvio padrão mostra uma dinâmica bem característica do crescimento microbiano. Nos primeiros momentos, ele é muito baixo porque todas as réplicas estão praticamente alinhadas e ainda não há variação perceptível. À medida que as culturas entram na fase exponencial, o desvio padrão aumenta bastante, refletindo diferenças individuais na velocidade de crescimento entre réplicas. Depois, quando o crescimento desacelera e as populações entram no plateau, o desvio padrão começa a cair novamente, indicando que os valores se estabilizam e as réplicas convergem para níveis finais semelhantes. Em resumo, a variabilidade é mínima no início, máxima durante o crescimento ativo e volta a reduzir na fase estacionária.

### Atribuição de estirpes por grupo

Cada grupo altera a variável `estirpe` na célula seguinte para o seu organismo:

| Grupo | Organismo | Tipo |
|---|---|---|
| 1 | `E_coli_K12` | Bactéria comensal |
| 2 | `Salmonella_enterica` | Bactéria patogénica |
| 3 | `L_acidophilus` | Bactéria probiótica |
| 4 | `C_difficile` | Bactéria patogénica |
| 5 | `S_cerevisiae` | Levedura comensal |
| 6 | `C_albicans` | Levedura patogénica |




#### **Questão**
Antes de correr o código, o que esperam? 
- Um patogénico cresce mais rápido que um comensal?
- Uma levedura cresce mais devagar que uma bactéria?

#### **Respostas**
* **Patogénicos vs comensais:** espera-se que um microrganismo patogénico cresça mais rápido, já que precisa competir e invadir o hospedeiro.
* **Leveduras vs bactérias:** espera-se que leveduras cresçam mais devagar que bactérias, porque o modo de divisão é diferente. As bactérias usam fissão binária, rápida e eficiente, enquanto as leveduras se dividem por gemulação, processo mais lento e energeticamente custoso. 

### 1.2 Filtrar os dados da vossa estirpe

Filtramos a dataframe de maneira a selecionar apenas as linhas correspondentes à estirpe escolhida, e removemos as colunas não numéricas (Strain, Replicate).

In [ ]:
# Colunas
colunas = df.columns.to_list()

In [ ]:
estirpe = "S_cerevisiae" # defina a estirpe a analisar

df_estirpe = df[df["Strain"] == estirpe] # filtrar o dataframe para a estirpe escolhida
#print(df_estirpe)

# ----Filtrar a dataframe para as colunas com dados apenas
df_estirpe_time = df_estirpe[2:]
print(df_estirpe_time)

#columns = XXX # LISTA o nome das colunas do dataframe
colunas = df_estirpe.columns.to_list()

time_cols = colunas[2:]  # mantém apenas as colunas de tempo (assumindo que as 2 primeiras são 'Strain' e 'Replicate')
#print(time_cols)

print(f"DataFrame filtrado para a estirpe {estirpe}:\n{df_estirpe.head(5)}")

### 1.3 Calcular média e desvio-padrão de todos os tempos

Para cada ponto temporal, calculamos a **média** e o **desvio-padrão** das 3 réplicas.

- A **média** dá-nos o valor central da medição
- O **desvio-padrão** indica a variabilidade entre réplicas — se é pequeno, as réplicas são consistentes


In [ ]:
# --- Usa a dataframe 'df_estirpe' para calcular os valores da média

#print(df_estirpe)

od_mean = df_estirpe[time_cols].mean().values # selecionar valores e calcular média para cada tempo
print(f"Médias calculadas por cada periodo de tempo analisado:\n{od_mean}")

#od_std  = XXX # calcular desvio-padrão para cada tempo

od_std = df_estirpe[time_cols].std().values
print(f"Desvios-padrão:\n{od_std}")


In [ ]:
time_array = np.array(time_cols, dtype=float) # criar array de tempos a das colunas de tempo
print(f"Array de tempos (horas): {time_array}")

#### **Questão:**

Olhando para a média dos valores, consegues antever:
- quando acaba a fase de adaptação do crescimento (lag)?
- quando começa  a fase de crescimento exponencial?
- quando se inicia a fase estacinária?

#### **Resposta:** Olhando para a evolução das médias ao longo do tempo, dá para identificar bem as fases do crescimento:

* Fase de adaptação (lag): os valores iniciais são muito baixos e praticamente constantes até cerca de 3 horas. Isso indica que as células ainda estão ajustando o metabolismo ao meio antes de iniciar a divisão.
* Fase exponencial: a partir de aproximadamente 4 horas, observa-se um aumento rápido e contínuo das médias, mostrando que as populações entraram em crescimento ativo.
* Fase estacionária: inicia-se por volta de 17h e as 20 horas, os valores começam a estabilizar e deixam de crescer de forma acentuada. Isso marca o início do plateau, quando os nutrientes se tornam limitantes e o crescimento se equilibra com a morte celular.

### 1.4 Representação gráfica

Vamos representar a curva de crescimento com a média ± desvio-padrão (barras de erro), mas também todas as réplicas individualmente.


In [ ]:
# --- Para o cálculo dos valores da média usa od_mean, od_std
# --- Para o cálculo dos valores das réplicas individuais usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'

fig, ax = plt.subplots(figsize=(8, 5))


# ────── Curva dos valores médios de crescimento
ax.plot(time_array, od_mean, #X = time, Y= média de OD em cada tempo
        "o-", color="#3266ad", linewidth=2, markersize=5, label='Média, n=3')

# ────── Curva dos valores de cada réplicas individual
print(df_estirpe)
for i, row in df_estirpe.iterrows():
    print(row)
    ax.plot(time_array, row[time_cols], #X = time, Y= valores de OD em CADA TEMPO
           "o-", linewidth=1, markersize=3, alpha=0.6, label=row["Replicate"]) # label usando o nome da réplica


# ────── Banda de desvio padrão
ax.fill_between(time_array, od_mean - od_std,  od_mean+od_std,  # X = time, Y_baixo = valor mais baixo (média - std), Y_alto = valor mais alto (média + std)
                color="#F96167", alpha=0.2, label="± SD")

ax.set_xlabel("Tempo (horas)")
ax.set_ylabel("OD600")
ax.set_title("Estimativa gráfica dos parâmetros")
ax.legend()

#mostra a imagem e fecha, permitindo a continua alteração da imagem
display(fig)
plt.close()

### **Questão**
* Conseguem identificar as fases lag, exponencial e estacionária?
* Como se comparam as diferentes réplicas individualmente com a média e desvio padrão?

### **Respostas**
* Sim, é possível identificar claramente as fases no gráfico:
* *Fase lag (adaptação)*: corresponde às primeiras horas (0–4h), em que a densidade óptica permanece baixa e quase constante. As células estão ajustando o metabolismo ao meio antes de iniciar a divisão.
* *Fase exponencial*: inicia-se por volta das 5–10h, quando a curva sobe de forma acentuada e contínua. É o período de crescimento ativo, com aumento rápido da densidade.
* *Fase estacionária*: começa após cerca de 16–20h, quando a curva se estabiliza em torno de OD600 ≈ 2.3. Aqui o crescimento se equilibra com a limitação de nutrientes e acúmulo de resíduos.

* **Quanto às réplicas:** As três réplicas (R1, R2, R3) acompanham bem a tendência da média, sem desvios extremos, isso nota-se no gráfico pela área sombreada (desvio padrão em relação à média) mostra que há alguma variabilidade, especialmente na fase exponencial, mas dentro de limites esperados. No plateau, todas convergem para valores muito próximos, confirmando consistência experimental.

# FASE 2 — Calcular os parâmetros de crescimento

Agora vamos passar da observação ao cálculo. Para cada estirpe, vamos estimar os 3 parâmetros de crescimento a partir dos dados:

| Passo | O que vamos fazer | Parâmetro | 
|-------|-------------------|-----------|
| 2.1 | Estimar o plateau (média dos últimos pontos) | **A** |
| 2.2 | Calcular a taxa máxima de crescimento (ΔOD/Δt) | **μ_max** |
| 2.3 | Estimar a fase lag a partir do ponto de inflexão | **λ** |

Cada grupo vai analisar **um microrganismo** e no final juntamos os resultados de todos.

### 2.1 Crescimento máximo — Plateau ou Fase Estacionária (A)

O plateau (A) representa a **densidade ótica máxima** que a cultura atinge — o ponto em que o crescimento estabiliza porque os nutrientes se esgotam ou os produtos tóxicos se acumulam.

Estimamos A como a **média dos últimos pontos temporais**, onde a curva já estabilizou.

**O que determina o valor de A?**
- **Capacidade de carga do meio:** quantidade de nutrientes disponíveis
- **Tolerância a produtos metabólicos:** ácidos, etanol, toxinas que se acumulam
- **Tamanho celular:** a mesma OD600 pode corresponder a números diferentes de células conforme o organismo (bactérias são mais pequenas que leveduras)

**Questão:** 
- Se duas estirpes crescem no mesmo meio, mas uma atinge um plateau mais alto, o que pode explicar a diferença?

* **Resposta:**


Quando duas estirpes crescem no mesmo meio mas uma atinge um plateau mais alto, isso significa que ela consegue acumular mais biomassa final. Essa diferença pode ser explicada pela maior eficiência metabólica na utilização dos nutrientes, pela capacidade de tolerar melhor os subprodutos tóxicos do crescimento, pela regulação genética que permite prolongar a fase de multiplicação antes da estabilização ou ainda por características estruturais das células que influenciam a densidade óptica medida. Em termos biológicos, um plateau mais elevado traduz uma vantagem adaptativa, mostrando que aquela estirpe consegue explorar melhor os recursos disponíveis e sustentar populações maiores em condições semelhantes.

In [ ]:
df_estirpe

In [ ]:
# --- Para o cálculo dos valores da média usa od_mean, od_std
# --- Para o cálculo dos valores das réplicas individuais usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'


# ─────────────── Plateau (A) ─────────────────────────────────────────
# ────── Calcular o plateu médio
valor_incial_A = 19 # em que valor a reta entra em A?
valores_A = od_mean[valor_incial_A:] # selecionar os valores a partir do tempo onde a curva parece estabilizar (usar o array de médias)
print(f"Valores selecionados: {valores_A}")

A_estimado = od_mean[19:].mean() # calcular a média dos valores para obter a estimativa do plateau
print(f"\nEstimativas gráficas para {estirpe}:")
print(f"  Plateau (A) média ≈ {A_estimado}")

#visualiza o valor médio do plateau no gráfico
ax.axhline(A_estimado, color="gray", linestyle="--", label=f"Plateau ≈ {A_estimado}") 
ax.legend()

# ────── Calcular o plateau para cada réplica individualmente
for i, row in df_estirpe.iterrows():      # loop por cada réplica usando df_estirpe
    od_rep = row[time_cols].values   # seleciona as colunas de valores → guarda os values de OD
    od_A_rep = od_rep[valor_incial_A:]                 # valores de A
    A_rep = od_A_rep.mean()                     # média → estimativa do plateau
    print(f"  {row['Replicate']}: Plateau ≈ {A_rep:.3f}")


display(fig)
plt.close()

### 2.2 Taxa de crescimento máxima (μ_max)

A taxa de crescimento máxima corresponde à **inclinação máxima** da curva — o momento em que as células se estão a dividir mais rapidamente (fase exponencial).

Calculamos μ_max como a derivada numérica da curva:

$$\mu_{max} = \max\left(\frac{\Delta OD}{\Delta t}\right)$$

Ou seja, para cada par de pontos consecutivos calculamos a variação de OD por hora (ΔOD/Δt) e ficamos com o **valor máximo**.

**O que influencia μ_max?**
- **Tipo de divisão celular:** bactérias (fissão binária, ~20 min) vs leveduras (gemulação, ~90 min)
- **Meio de cultura:** disponibilidade de nutrientes, temperatura, pH
- **Metabolismo:** aeróbio vs anaeróbio, fermentativo vs respiratório


In [ ]:
# --- Para o cálculo dos valores da média usa od_mean, od_std
# --- Para o cálculo dos valores das réplicas individuais usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'


# ─────────────── Taxa máxima (mu) ────────────────────────────────────

# ────── Calcular o mu médio
# -- np (numpy é usado para trabalhar com arrays e fazer operações amtemáticas)
dod = np.diff(od_mean) # calcular a diferença entre os valores de OD para obter a taxa de crescimento
print(f"Taxa de crescimento (ΔOD):{dod}")

dt = np.diff(time_array) # calcular intervalo de tempo
print(f"Intervalos de tempo (Δt): {dt}")

dod_dt = dod/dt # aplicar a fórmula da taxa de crescimento (ΔOD/Δt)
print(f"Taxa de crescimento por hora (ΔOD/Δt): {dod_dt}")

mu_estimado  = dod_dt.max() # encontrar o valor máximo de μ_max no ARRAY de valores
print(f"Taxa de crescimento:{mu_estimado}")

idx_max = np.argmax(dod_dt) # encontrar o índice no array no valor máximo
time_mu_max  = time_array[idx_max] # para identificar o tempo correspondente
print(f"O crescimento máximo de {mu_estimado} OD/h ocorre no tempo de {time_mu_max} horas")

# ────── Calcular mu para cada réplica individualmente
for i, row in df_estirpe.iterrows():
    od_rep      = row[time_cols].values # extrair os valores de OD da réplica
    dod_dt_rep  = np.diff(od_rep)/dt # calcular a taxa de crescimento da réplica
    mu_rep      = dod_dt_rep.max() # estimar μ_max da réplica
    print(f"  {df_estirpe.loc[i, 'Replicate']}: μ_max ≈ {mu_rep} OD/h")


# --- plot no gráfico
ax.plot(time_array[idx_max] +1, dod_dt[idx_max], "o", color="red", 
        label=f"μ_max ≈ {mu_estimado} OD/h")
ax.legend()

print(f"Estimativas gráficas para {estirpe}:")
print(f"  Plateau (A)  ≈ {A_estimado:.3f}")
print(f"  μ_max        ≈ {mu_estimado:.3f} OD/h")

display(fig)
plt.close()

### 2.3 Estimativa da Fase Lag (λ)

A fase lag é calculada a partir do ponto de inflexão da curva:

$$\lambda = t_{inf} - \frac{OD_{inf} - OD_0}{\mu_{max}}$$

Onde:
- **lambda** — tempo da fase lag (se λ = 2 horas, significa que a cultura ficou cerca de 2 horas em adaptação antes de crescer rapidamente)
- **t_inf** — tempo no ponto de inflexão (onde o crescimento é mais rápido)
- **OD_inf** — valor de OD nesse ponto
- **OD_0** — valor de OD no tempo zero
- **μ_max** — taxa de crescimento máxima (calculada no passo anterior)

Ouuuu....

Podemos estimar quando a OD ultrapassa o valor de inicial.

#### **Questão:** 
- Um organismo com uma fase lag longa é vantajoso ou desvantajoso? Depende do contexto?

#### **Resposta**

Um organismo com uma fase lag longa pode ser visto como vantajoso ou desvantajoso dependendo do contexto. Em ambientes competitivos, onde os nutrientes são abundantes mas disputados, uma lag prolongada é desvantajosa porque o organismo demora a iniciar o crescimento e pode ser ultrapassado por outros que crescem mais rápido. Já em ambientes hostis ou variáveis, uma lag longa pode ser vantajosa porque reflete um período de adaptação mais cuidadoso: o microrganismo ajusta o metabolismo, ativa mecanismos de defesa e prepara-se melhor para sobreviver.

In [ ]:
# --- Para o cálculo dos valores da média usa od_mean, od_std
# --- Para o cálculo dos valores das réplicas individuais usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'

# ────── Calcular lam da média
threshold = od_mean[0] + 0.1 # OD inicial + margem para ignorar ruído
print(f"Limite de crescimento em fase Lag: {threshold}")

array_valores_acima_threshold = od_mean > threshold # cria um array booleano (True/False) dos valores de od_mean acima do threshold defnido
print(f"Valores acima do threshold: {array_valores_acima_threshold}")

valores_tempo_acima_threshold = time_array[array_valores_acima_threshold] # Retira os valores de tempo onde True
print(f"Tempos acima do crescimento lag: {valores_tempo_acima_threshold}")

lam_estimado = valores_tempo_acima_threshold[0] # primeiro tempo onde OD ultrapassa o threshold
print(f"Tempo que o crescimento sai de fase lag: {lam_estimado}")


In [ ]:
 # ────── Calcular Lag para cada réplica individualmente
for i, row in df_estirpe.iterrows():
    lam_rep = row[time_cols].values   # valores da réplica
    acima_threshold = lam_rep > threshold   # comparar com threshold
    tempo_acima_threshold = time_array[acima_threshold]  # tempos acima do threshold
    lam_estimado = tempo_acima_threshold[0]
    print(f"{df_estirpe.loc[i, "Replicate"]}: lag = {lam_estimado} h")
    ax.axvline(lam_estimado, color="orange", linestyle="--", label=f"Lag ≈ {lam_estimado}h") # representar graficamente
    ax.legend()

print(f"Estimativas gráficas para {estirpe}:")
print(f"  Plateau (A)  ≈ {A_estimado}")
print(f"  μ_max        ≈ {mu_estimado:.4f} OD/h")
print(f"  Fase lag     ≈ {lam_estimado} h")

display(fig)
plt.close()

# FASE 3 — Comparar o crescimento das diferentes estirpes

### 3.1 Tabela resumo dos parâmetros de crescimento

Depois de estimados os parâmetros para todas as estirpes, compilamos os resultados numa tabela com a **média ± desvio-padrão** (3 réplicas) de cada parâmetro:

| Parâmetro | O que representa | Como foi estimado |
|---|---|---|
| **A** (mean ± sd) | Plateau — densidade máxima atingida | Média dos últimos 5 pontos temporais |
| **μ_max** (mean ± sd) | Taxa de crescimento máxima | Máximo de ΔOD/Δt na fase exponencial |
| **λ** (mean ± sd) | Fase lag — tempo até iniciar crescimento | Fórmula: λ = t_inf − (OD_inf − OD₀) / μ_max |

A tabela resultante terá este formato — **uma linha por estirpe, com média e desvio-padrão de cada parâmetro**:

| Strain | Replicate | A_mean | A_sd | mu_mean | mu_sd | lam_mean | lam_sd |
|---|---|---|---|---|---|---|---|
| C_albicans | ... | ... | ... | ... | ... | ... | ... |
| C_difficile | ... | ... | ... | ... | ... | ... | ... |
| E_coli_K12 | ... | ... | ... | ... | ... | ... | ... |
| L_acidophilus | ... | ... | ... | ... | ... | ... | ... |
| S_cerevisiae | ... | ... | ... | ... | ... | ... | ... |
| Salmonella_enterica | ... | ... | ... | ... | ... | ... | ... |



In [ ]:
# --- Para o cálculo de todos os dados usa a dataframe 'df_estirpe' e colunas de dados 'time_cols'

#────── Criar dataframe vazia para guardar os resultados
df_resultados = pd.DataFrame(columns = ["Strain", "Replicate", "A", "mu", "lam"])

#────── Criar função para calcular os diferentes parâmetros de crescimento
def calcular_parametros(t, od): # cria função para calcular os parÂmetros de crescimento
    
    # Plateau — média dos últimos 20% dos pontos
    n_plateau = int(0.2 * len(od))
    A = od[-n_plateau:].mean()

    # Derivada númerica (diferenças finitas)
    dod_dt = np.diff(od) / np.diff(t)

    #Mu_max-- valor máximo da derivada
    idx_max = np.argmax(dod_dt)
    mu = dod_dt[idx_max]

   
    # Fase lag — threshold
    threshold = od[0] + 0.1
    lam = t[od> threshold][0]

    return A, mu, lam

#────── Calcular e armazenar os parâmetros de crescimento para todas as estirpes
for estirpe in df["Strain"].unique(): # iterar por estirpe
    df_estirpe = df[df["Strain"]== estirpe] # filtrar por estirpe
    print(f"imprimindo o df strip time \n{df_estirpe}")
    df_estirpe_time = df_estirpe[2:] # manter apenas colunas com valores de crescimento
    print(f"imprimindo o df strip time \n{df_estirpe_time}")

    for i, row in df_estirpe.iterrows(): # iterar por réplica

        od_rep = row[time_cols].values # valores de OD da linha 
        A, mu, lam = calcular_parametros(time_array, od_rep)

        #guardar resultados
        df_resultados.loc[i, "Strain"]    = estirpe
        df_resultados.loc[i, "Replicate"] = df_estirpe.loc[i, "Replicate"]
        df_resultados.loc[i, "A"] = A
        df_resultados.loc[i, "mu"] = mu
        df_resultados.loc[i, "lam"] = lam


print("Parâmetros por réplica:")
print(df_resultados)

#────── Colapsar em valores médios, calculando o desvio padrão
df_resumo = df_resultados.groupby("Strain").agg(  #agrupar as diferentes linhas pela coluna strain, agregando...
    A_mean=("A", "mean"), A_sd=("A", "std"), # coluna A_mean em que agregamos os valores de A pela média e coluna A_std em que agregamos os valores de A pelo desvio padrão
    mu_mean=("mu", "mean"), mu_sd=("mu", "std"), # repete para mu
    lam_mean=("lam", "mean"), lam_sd=("lam", "std"),# repete para lam
).round(3)
print("\nResumo (média ± SD por estirpe):")
print(df_resumo)

#### **Questão:** 
- Qual a estirpe com maior variabilidade (SD alto) em cada parâmetro? O que pode explicar isso?

#### **Resposta** 

A estirpe com maior variabilidade no plateau e na fase lag é C. difficile, mostrando comportamento menos previsível na adaptação e na densidade final. Já em taxa máxima de crescimento, quem apresenta maior dispersão é E. coli K12, provavelmente porque cresce muito rápido e pequenas diferenças entre réplicas se amplificam. Essas variações refletem tanto características fisiológicas próprias de cada microrganismo quanto a sensibilidade às condições experimentais.

### 3.1 Simulação das curvas de crescimento — Modelo de Gompertz

Tendo os 3 parâmetros estimados (A, μ_max, λ) para cada estirpe, podemos reconstruir a curva de crescimento completa usando o **modelo de Gompertz modificado**:

$$OD(t) = A \cdot \exp\left(-\exp\left(\frac{\mu_{max} \cdot e}{A} \cdot (\lambda - t) + 1\right)\right)$$

Isto permite-nos **sobrepor todas as estirpes no mesmo gráfico** e comparar visualmente.

| Parâmetro | Símbolo | Unidade | Significado biológico |
|---|---|---|---|
| Medida de Crescimento | OD(t) | h | OD da população de microrganismos no tempo \(t\)|
| Tempo | t | h | Variável independente — o tempo decorrido desde o início do ensaio. |
| Número de Euler | e | — | Constante matemática (≈ 2,718).|

Fórmula:
- A estrutura é uma **exponencial dupla** (exp dentro de exp), o que gera a forma sigmoide assimétrica característica.
- O termo **e** na fórmula é o número de Euler (≈ 2,718), e aparece como fator de escala que garante que μ_max corresponda exatamente à taxa máxima de crescimento — é uma consequência da derivação matemática do modelo, não um parâmetro ajustável.
- O argumento interno, **(μ_max · e / A) · (λ − t) + 1**, controla a transição entre as fases.
- Quando **t ≪ λ**, (λ − t) é grande e positivo, a exponencial interna é muito grande, e a exponencial externa tende a zero — a população ainda não cresceu.
- Quando **t ≫ λ**, (λ − t) torna-se muito negativo, a exponencial interna tende a zero, e OD(t) tende a A — a população atingiu o máximo.
- A transição entre estes extremos dá-nos a **fase exponencial**.

In [ ]:
# Para a simulação usa a dataframe df_resumo
print(df_resumo)

# ────── Cria o gráfico
fig_all, ax_all = plt.subplots(figsize=(10, 6))

# ────── Função que faz a simulação da curva por estirpe
def gompertz(t, A, mu, lam):
    return A*np.exp(-np.exp((mu * np.e / A) * (lam - t) + 1)) #Tenta completar a fórmula. Lembra-te que o np permite operações matemáticas (np.exp, np.e)
    
# ---np.linspace(start, stop, num) — cria um array de num valores igualmente espaçados entre start e stop.
t_curva = np.linspace(0, time_array[-1], 200)
print(f"Linha temporal criada: {t_curva}")

# ──────simula a curva por cada estirpe
for estirpe in df_resumo.index.unique():     #onde estão as estirpes detalhadas?
    params   = df_resumo.loc[estirpe]      #retira os valores da estirpe da df_resumo
    print(params)                            #retira os valores da estirpe da df_resumo
    od_curva = gompertz(t_curva, params["A_mean"], params["mu_mean"], params["lam_mean"])       #insere os valores necessários para a função
    ax_all.plot(t_curva, od_curva, linewidth=2, label=estirpe.replace("_", " "))    #plota a linha no gráfico - X = tempo, Y = ods calculados por gompertz

ax_all.set_xlabel("Tempo (horas)")
ax_all.set_ylabel("OD600")
ax_all.legend()
ax_all.grid(alpha=0.3)
plt.tight_layout()
plt.show()

#### **Questão:** 

- Quem arranca primeiro? (λ mais curto = curva desloca-se para a esquerda)
* **Resposta:** E. Coli K12
- Quem cresce mais rápido? (μ_max maior = curva mais íngreme)
* **Resposta:** E. Coli K12
- Quem atinge maior densidade? (A maior = plateau mais alto)
* **Resposta:** E. Coli K12
- Olhando para o gráfico, consigam identificar os dois grandes grupos (bactérias vs leveduras). O que os distingue?
* **Resposta:** Bactérias

### 3.2 Visualização comparativa de parâmetros de crescimento

Para facilitar a comparação dos parâmetros de crescimento, é possivel criar visualização de gráfico de barras de todas as estirpes.

In [ ]:
#  Para a visualização usa a dataframe 'df_resumo'

print(df_resumo)

# ──────Cria figura com espaço para 3 gráficos
fig_bar, ax_bar = plt.subplots(1, 3, figsize=(14, 5))

# --- No params_plot definimos que o que queremos ver em cada gráfico
# --- mean_col, sd_col, titulo, unidade 
params_plot = [
    ("A_mean",   "A_sd",   "Plateau (A)",    "OD600"),
    ("mu_mean",  "mu_sd",  "Taxa máxima (µ)", "OD/h"),
    ("lam_mean", "lam_sd", "Fase lag (λ)",    "horas"),
]

estirpes = df_resumo.index.to_list() # retira as estirpes da df_resumo - index
print(f"lista de estirpes: {estirpes}")

# ────── Criar um gráfico de barras para cada parâmetro e cada estirpe
# --- O zip vai retirar os todos os valores em ax_bar e params_plot
for ax, (mean_col, sd_col, titulo, unidade) in zip(ax_bar, params_plot):
    valores = df_resumo[mean_col].values #Retira da df_resumo os values da média - media_col
    erros   = df_resumo[sd_col].values   #Retira da df_resumo os values da média - sd_col

    bars = ax.bar(estirpes, valores, yerr=erros) # X = estirpes, Y = valores de média, yerr = valores para as barras de erro

    ax.set_xticks(range(len(estirpes)))
    ax.set_xticklabels(estirpes, fontsize=8, rotation=45, ha="right")
    ax.set_title(titulo)
    ax.set_ylabel(unidade)


plt.suptitle("Comparação de parâmetros de crescimento (média ± SD)", fontsize=13)
plt.tight_layout()
plt.show()

#### **Questão:** 
- Comparem A com μ_max: os organismos que crescem mais rápido são os que atingem maior densidade?
#### **Resposta:** 

Sim, há uma correlação positiva entre velocidade de crescimento e densidade final, embora não seja absoluta.

### 3.3 Os parâmetros de crescimento são estatisticamente diferentes entre estirpes?

Visualmente, as curvas e gráficos de barras parecem diferentes — mas será que essas diferenças são **estatisticamente significativas** ou podem ser apenas variabilidade experimental?

Para responder, vamos aplicar:

1. **ANOVA One-way** — testa se existe pelo menos uma estirpe diferente das outras (para cada parâmetro)
2. **Tukey HSD (Honestly Significant Difference)** (post-hoc) — se a ANOVA for significativa, identifica **quais pares** de estirpes são diferentes

Relembrando:
- **p < 0.05** → diferença significativa (rejeitamos H₀)
- **p ≥ 0.05** → sem evidência de diferença (não rejeitamos H₀)


In [ ]:
# Valores a usar estão na dataframe 'df_resultados' e a lista de estirpes na variável 'estirpe'

print(df_resultados)
print(estirpe)

for param, titulo in [("A", "Plateau (A)"), ("mu", "Taxa máxima (µ)"), ("lam", "Fase lag (λ)")]:

    # ── Preparar Valores ────────────────────
    # criar um dicionário com os valores do parâmetro por estirpe
    grupos = {}

    for e in estirpes:
        df_estirpe_params = df_resultados[df_resultados["Strain"] == e]  # filtrar para a estirpe
        grupos[e] = df_estirpe_params[param].values.astype(float) # array para o parÂmetro (já tens a variável no primeiro for loop, com os 3 values das réplicas e converte em float (.astype)
    print(f"Grupos: {grupos}")

    # ── ANOVA entre todas as estirpes ────────────────────
    f_stat, p_anova = stats.f_oneway(*grupos.values()) # faz a ANOVA entre os grupos usando a função f_oneway do scipy.stats -
    # --- *grupos = stats.f_oneway(grupos["E_coli_K12"], grupos["Salmonella_enterica"], grupos["L_acidophilus"], ...)
    #grupos.values(): [1.5065  1.51975 1.58   ] [2.39325 2.00575 2.15375] [2.87975 2.7845  2.7195 ] [1.89175 2.16425 1.93875] [2.21225 2.167   2.259  ] [2.55375 2.54975 2.77125]
    print(f"\nANOVA (todas as estirpes): F={f_stat}, p={p_anova}")


    # ── Tukey HSD post-hoc ───────────────────────────────
    valores = np.concatenate(list(grupos.values()))
    #  --- np.concatenate(list(grupos.values())) - [1.5065  1.51975 1.58    2.39325 2.00575 2.15375 2.87975 2.7845  ....
    labels = np.repeat(list(grupos.keys()), 3) # diz que cada 3 valores são a réplica de uma espécie
    tukey   = pairwise_tukeyhsd(valores, labels, alpha=0.05)
    print("\nTukey HSD:")
    print(tukey.summary())

#### **Questões**

- Para qual dos 3 parâmetros (A, μ, λ) existem mais diferenças significativas entre estirpes?
- Há estirpes que não são significativamente diferentes entre si? Quais? Faz sentido biologicamente?
- *E. coli* e *Salmonella* (ambas Enterobacteriaceae) têm parâmetros semelhantes? E *S. cerevisiae* e *C. albicans* (ambas leveduras)?

#### **Respostas**
* **μ (taxa máxima de crescimento):** apresenta o maior número de comparações significativas entre estirpes, ao passo que A (plateau), apresenta diferenças moderadas e a λ (fase lag) → menos diferenças significativas, várias comparações não rejeitam H0. logo, conlui-se que o parâmetro μ é o mais discriminante entre espécies.
* **Há sim:** A E. coli vs Salmonella → não diferem em A nem em λ; S. cerevisiae vs C. albicans → não diferem em A nem em λ e outros pares como C. difficile vs L. acidophilus também não mostraram diferenças em alguns parâmetros.

*Sentido biologico:* A E. coli e Salmonella são Enterobacteriaceae, fisiologia semelhante; ao passo que S. cerevisiae e C. albicans são leveduras, ambas eucariotas, com metabolismo fermentativo próximo.

* **Comparação entre grupos próximos** As Enterobacteriaceae (E. coli & Salmonella):são semelhantes em A e λ → atingem densidade final e fase lag parecidas e diferem em μ → pequenas variações na velocidade de crescimento.

* Leveduras (S. cerevisiae & C. albicans): Semelhantes em A e λ → comportamento geral próximo, mas diferenças em μ → S. cerevisiae cresce mais rápido em glicose, C. albicans adapta-se ao hospedeiro.

### 3.4 Os parâmetros de crescimento são estatisticamente diferentes entre Bactérias e Leveduras

Agora agrupamos os organismos por **tipo** (bactérias vs leveduras) e testamos se há diferenças entre os dois grupos.

**Teste t de Student** — compara as médias de **dois grupos** para determinar se a diferença entre eles é estatisticamente significativa (p < 0,05).


In [ ]:
# ══════════════════════════════════════════════════════════
# ANÁLISE POR GRUPO — bactérias vs leveduras
# ══════════════════════════════════════════════════════════
bacterias = ["E_coli_K12", "Salmonella_enterica", "L_acidophilus", "C_difficile"]
leveduras = ["S_cerevisiae", "C_albicans"]

for param, titulo in [("A", "Plateau (A)"), ("mu", "Taxa máxima (µ)"), ("lam", "Fase lag (λ)")]:

    # criar um dicionário com os valores do parâmetro por estirpe
    grupos = {}
    for e in estirpes:
        df_estirpe_params = df_resultados[df_resultados["Strain"] == e]  # filtrar para a estirpe
        grupos[e] = df_estirpe_params[param].values.astype(float)  # array com os 3 valores das réplicas

    #DIFERENÇA
    # juntar todos os valores de bactérias num único array
    vals_bact = np.concatenate([grupos[e] for e in bacterias])
    print(f'ex: vals_bact variável: {vals_bact}')
    # juntar todos os valores de leveduras num único array
    vals_lev  = np.concatenate([grupos[e] for e in leveduras])

    # ── t-test bactérias vs leveduras ────────────────────
    # -- ttest_ind(valores de crescimento das bacterias, valores de crescimento das leveduras)
    t_stat, p_ttest = stats.ttest_ind(vals_bact, vals_lev)
    print(f"\nt-test bactérias vs leveduras para {titulo}: t={t_stat:.3f}, p={p_ttest:.4f}")
    print(f"  bactérias: média={vals_bact.mean():.3f} ± {vals_bact.std():.3f}")
    print(f"  leveduras: média={vals_lev.mean():.3f} ± {vals_lev.std():.3f}")

#### **Questões**

- As bactérias atingem um plateau mais alto que as leveduras? É significativo?
- As leveduras demoram mais a iniciar o crescimento? O que pode explicar isso biologicamente?
- A taxa de crescimento é diferente? As bactérias dividem-se por **fissão binária** (rápido) e leveduras por **gemulação** (mais lento)


#### **Respostas**
* Sim, bactérias atingem um plateau mais alto que leveduras e é significativo (Bactérias: média ≈ 2.40, Leveduras: média ≈ 1.87,significância (p = 0.0103)).

* **Leveduras demoram mais a iniciar o crescimento**, isto porque as leveduras precisam ajustar metabolismo e maquinaria celular eucariótica antes de dividir, enquanto bactérias adaptam-se mais rapidamente ao meio.
* **Bactérias crescem mais rápido:** isto é notório porque a divisão por fissão binária é mais eficiente e rápida; leveduras dividem por gemulação, processo mais lento e energeticamente custoso.

### 3.4 Os parâmetros de crescimento são estatisticamente diferentes entre Patogénicas vs Comensais

Será que os organismos **patogénicos** crescem de forma diferente dos **comensais**? Separamos a análise para bactérias e leveduras.

| Bactérias patogénicas | Bactérias comensais | Leveduras patogénicas | Leveduras comensais |
|---|---|---|---|
| *Salmonella*, *C. difficile* | *E. coli* K12, *L. acidophilus* | *C. albicans* | *S. cerevisiae* |


In [ ]:
# ── Bactérias patogénicas vs comensais ──────────────────
bact_patogenicas = ["Salmonella_enterica", "C_difficile"]
bact_comensais   = ["E_coli_K12", "L_acidophilus"]
# ── Leveduras patogénicas vs comensais ──────────────────
lev_patogenicas  = ["C_albicans"]
lev_comensais    = ["S_cerevisiae"]

for param, titulo in [("A", "Plateau (A)"), ("mu", "Taxa máxima (µ)"), ("lam", "Fase lag (λ)")]:
    
    #separador visual
    print(f"\n{'═'*55}")
    print(f"{titulo}")
    print(f"{'═'*55}")

    # criar um dicionário com os valores do parâmetro por estirpe
    grupos = {}
    for e in estirpes: #loop pelas estirpes
        df_estirpe_params = df_resultados[df_resultados["Strain"] == e]  # filtrar para a estirpe
        grupos[e] = df_estirpe_params[param].values.astype(float)        # array com os 3 valores das réplicas

    # ── Bactérias patogénicas vs comensais ───────────────
    vals_bact_pat = np.concatenate([grupos[e] for e in bact_patogenicas])     # juntar valores das patogénicas
    vals_bact_com = np.concatenate([grupos[e] for e in bact_comensais])     # juntar valores das comensais
    t_stat, p_ttest = stats.ttest_ind(vals_bact_pat, vals_bact_com)   # t-test
    print(f"\nt-test bactérias patogénicas vs comensais para {titulo}: t={t_stat:.3f}, p={p_ttest:.3f}")
    print(f"  patogénicas: média={vals_bact_pat.mean():.3f} ± {vals_bact_pat.std():.3f}")
    print(f"  comensais:   média={vals_bact_com.mean():.3f} ± {vals_bact_com.std():.3f}")

    # ── Leveduras patogénicas vs comensais ───────────────
    vals_lev_pat = np.concatenate([grupos[e] for e in lev_patogenicas])      # juntar valores das patogénicas
    vals_lev_com = np.concatenate([grupos[e] for e in lev_comensais])      # juntar valores das comensais
    t_stat, p_ttest = stats.ttest_ind(vals_lev_pat, vals_lev_com)    # t-test
    print(f"\nt-test leveduras patogénicas vs comensais para {titulo}: t={t_stat:.3f}, p={p_ttest:.3f}")
    print(f"  patogénicas: média={vals_lev_pat.mean():.3f} ± {vals_lev_pat.std():.3f}")
    print(f"  comensais:   média={vals_lev_com.mean():.3f} ± {vals_lev_com.std():.3f}")

#### **Questões**

- Há diferenças significativas entre patogénicos e comensais dentro das bactérias? E dentro das leveduras?
- Se *C. albicans* (patogénica) cresce mais devagar que *S. cerevisiae* (comensal), que importância pode ter?

#### **Respostas**

* **Diferenças dentro das bactérias:** 
* Plateau (A): patogénicas (2.465) vs comensais (2.396) → p = 0.971, não significativo;
* Taxa máxima (μ): patogénicas (1.052) vs comensais (1.269) → p = 0.572, não significativo.
* Fase lag (λ): patogénicas (3.33) vs comensais (3.83) → p = 0.654, não significativo.
* **Conclusão:** Nas bactérias, não há diferenças significativas entre bactérias patogénicas e comensais.

* **Diferenças dentro das leveduras**
* Plateau (A): patogénicas (1.535) vs comensais (2.213) → p < 0.001, significativo.
* Taxa máxima (μ): patogénicas (0.257) vs comensais (0.421) → p = 0.012, significativo.
* Fase lag (λ): patogénicas (8.0) vs comensais (6.67) → p = 0.016, significativo.
* **Conclusão:** As Leveduras patogénicas diferem claramente das comensais em todos os parâmetros.

* **Importância biológica de a C. albicans crescer mais devagar que S. cerevisiae:** A C. albicans (patogénica) cresce mais devagar e com fase lag mais longa, isso pode ser vantajoso para patogenicidade, porque permite adaptação ao hospedeiro antes de iniciar crescimento. Além do mais, isto serve como uma estratégia de sobrevivência em ambientes hostis (sistema imune, competição com outros agentes), ao passo que a S. cerevisiae (comensal/industrial) cresce rápido em meios ricos, mas não precisa dessa adaptação.

# FASE 4 — Conclusões

Respondam em grupo às seguintes questões:

1. **Ranking de crescimento:** Ordenem os 6 organismos do mais rápido ao mais lento. Coincidem com o que esperavam?

2. **Bactérias vs leveduras:** Quais as principais diferenças? São estatisticamente significativas para todos os parâmetros?

3. **Patogénicos vs comensais:** A patogenicidade está associada a um padrão de crescimento diferente?

4. **Previsão para co-cultura:** Com base nestes resultados, o que esperam que aconteça quando os 6 organismos crescerem juntos?
   - Quem vai dominar?
   - Quem pode ser eliminado?
   - Que mecanismos de interação podem ocorrer? (competição, produção de bacteriocinas, alteração de pH...)

---

**Na próxima aula:** Vamos usar **RT-PCR** com primers específicos para verificar a presença de cada espécie numa co-cultura real — e ver se as vossas previsões estavam corretas!
